# EDA: FIFA World Cup #

Este notebook invoca las clases `CargadorDatos`, `GestorPartidos` y `ProcesadorEDA`  para hacer la ingesta, limpieza y análisis exploratorio de los partidos de la FIFA World Cup.

## 1. Importación de librerías y clases

Agregamos la carpeta `src` al path de Python para poder importar nuestras clases desde el notebook.

In [13]:
import sys
sys.path.append("../src")

import pandas as pd
pd.set_option("display.max_columns", None)

from ingesta.cargador_datos import CargadorDatos
from eda.procesador_eda import ProcesadorEDA
from gestor.gestor_partidos import GestorPartidos

## 2. Ingesta de datos

Descargamos el CSV público de partidos internacionales y filtramos únicamente los partidos de la FIFA World Cup usando la clase `CargadorDatos`.

In [14]:
cargador = CargadorDatos(url_origen="https://raw.githubusercontent.com/martj42/international_results/master/results.csv", ruta_raw="../data/raw/partidos-mundial.csv", ruta_processed="../data/processed/partidos-mundial-procesado.csv",)
df_crudo = cargador.descargar_y_filtrar()

print(f"Partidos de Copa Mundial: {len(df_crudo)}")
df_crudo.head()

Partidos de Copa Mundial: 1058


,id_partido,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1,2,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True
2,3,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True
3,4,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
4,5,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True


## 3. Limpieza de datos y columnas derivadas

Usamos `ProcesadorEDA` para eliminar partidos sin marcador (aún no jugados) y agregar columnas derivadas: año, total de goles, diferencia de goles y ganador.

In [15]:
procesador = ProcesadorEDA(df_crudo)
procesador.limpieza_datos()
df_procesado = procesador.agregar_columnas_derivadas()

print(f"Partidos tras limpieza: {len(df_procesado)}")
df_procesado.head()

Partidos tras limpieza: 1049


,id_partido,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,anio,total_goles,diferencia_goles,ganador
0,1,1930-07-13,Belgium,United States,0,3,FIFA World Cup,Montevideo,Uruguay,True,1930,3,-3,Visitante
1,2,1930-07-13,France,Mexico,4,1,FIFA World Cup,Montevideo,Uruguay,True,1930,5,3,Local
2,3,1930-07-14,Brazil,Yugoslavia,1,2,FIFA World Cup,Montevideo,Uruguay,True,1930,3,-1,Visitante
3,4,1930-07-14,Peru,Romania,1,3,FIFA World Cup,Montevideo,Uruguay,True,1930,4,-2,Visitante
4,5,1930-07-15,Argentina,France,1,0,FIFA World Cup,Montevideo,Uruguay,True,1930,1,1,Local


## 4. Persistencia del dataset procesado

Guardamos el dataset limpio y enriquecido en `data/processed/`, en CSV y JSON, usando el método `guardar_procesado` de `CargadorDatos`.

In [16]:
cargador.guardar_procesado(df_procesado)
print("Dataset procesado guardado en data/processed/")

Dataset procesado guardado en data/processed/


## 5. Resumen descriptivo

Estadísticas generales (media, desviación, mínimo, máximo, cuartiles) de las variables numéricas del partido.

In [17]:
procesador.resumen_descriptivo()

,home_score,away_score,total_goles,diferencia_goles
count,1049.000000,1049.000000,1049.000000,1049.000000
mean,1.587226,1.243089,2.830315,0.344137
std,1.498813,1.286680,1.914637,2.034241
min,0.000000,0.000000,0.000000,-8.000000
25%,1.000000,0.000000,1.000000,-1.000000
50%,1.000000,1.000000,3.000000,0.000000
75%,2.000000,2.000000,4.000000,1.000000
max,10.000000,8.000000,12.000000,9.000000


## 6. Matriz de correlación

Esta parte nos permite ver qué tan relacionadas están las variables numéricas entre sí.

In [18]:
procesador.matriz_correlacion()

,home_score,away_score,total_goles,diferencia_goles,anio
home_score,1.000000,-0.061227,0.741672,0.775519,-0.122333
away_score,-0.061227,1.000000,0.624094,-0.677623,-0.160501
total_goles,0.741672,0.624094,1.000000,0.151712,-0.203625
diferencia_goles,0.775519,-0.677623,0.151712,1.000000,0.011384
anio,-0.122333,-0.160501,-0.203625,0.011384,1.000000


## 7. ¿El país sede gana más?

Aqui vamos a comparar el porcentaje de victorias de un equipo cuando juega como país anfitrión (en su propio país) contra el resto de los locales.

In [19]:
procesador.desempeno_pais_sede()

,condicion,pct_victorias
0,Anfitrión jugando en casa,62.12
1,Resto de locales (no anfitriones),43.51


## 8. Detección de outliers: goleadas históricas atípicas

Ahora vamos a usar el criterio de rango intercuartílico (IQR) para identificar los partidos con una cantidad de goles fuera de lo común.

In [20]:
outliers = procesador.detectar_outliers_goles()
outliers[["date", "home_team", "away_team", "home_score", "away_score", "total_goles"]]

,date,home_team,away_team,home_score,away_score,total_goles
94,1954-06-26,Switzerland,Austria,5,7,12
36,1938-06-05,Brazil,Poland,6,5,11
312,1982-06-15,Hungary,El Salvador,10,1,11
88,1954-06-20,Germany,Hungary,3,8,11
105,1958-06-08,France,Paraguay,7,3,10
9,1930-07-19,Argentina,Mexico,6,3,9
91,1954-06-23,Germany,Turkey,7,2,9
79,1954-06-17,Hungary,South Korea,9,0,9
134,1958-06-28,France,Germany,6,3,9
240,1974-06-18,Yugoslavia,DR Congo,9,0,9


## 9. Consultas con GestorPartidos

Aqui se creó el gestor a partir del dataset ya procesado, y exploramos algunas consultas de ejemplo: ventaja de local, partidos de un equipo, partidos de una edición específica, y el ranking de diferencia de goles.

In [21]:
gestor = GestorPartidos(df_procesado)

gestor.ventaja_local()

{'victorias_local': 481,
 'victorias_visitante': 332,
 'empates': 236,
 'pct_victorias_local': np.float64(45.85)}

### Partidos de un equipo específico

In [22]:
gestor.get_por_equipo("Costa Rica")[["date", "home_team", "away_team", "home_score", "away_score", "ganador"]]

,date,home_team,away_team,home_score,away_score,ganador
419,1990-06-11,Costa Rica,Scotland,1,0,Local
430,1990-06-16,Brazil,Costa Rica,1,0,Local
442,1990-06-20,Sweden,Costa Rica,1,2,Visitante
449,1990-06-23,Czechoslovakia,Costa Rica,4,1,Local
591,2002-06-04,China,Costa Rica,0,2,Visitante
606,2002-06-09,Costa Rica,Turkey,1,1,Empate
623,2002-06-13,Costa Rica,Brazil,2,5,Visitante
644,2006-06-09,Germany,Costa Rica,4,2,Local
661,2006-06-15,Ecuador,Costa Rica,3,0,Local
676,2006-06-20,Costa Rica,Poland,1,2,Visitante


### Partidos de una edición específica del Mundial

In [23]:
gestor.get_por_anio(2014)[["date", "home_team", "away_team", "home_score", "away_score"]]

,date,home_team,away_team,home_score,away_score
772,2014-06-12,Brazil,Croatia,3,1
773,2014-06-13,Chile,Australia,3,1
774,2014-06-13,Mexico,Cameroon,1,0
775,2014-06-13,Spain,Netherlands,1,5
776,2014-06-14,Colombia,Greece,3,0
...,...,...,...,...,...
831,2014-07-05,Netherlands,Costa Rica,0,0
832,2014-07-08,Brazil,Germany,1,7
833,2014-07-09,Netherlands,Argentina,0,0
834,2014-07-12,Brazil,Netherlands,0,3


### Top 10 selecciones por diferencia de goles histórica

In [24]:
tabla_diferencia = gestor.tabla_diferencia_goles(top_n=10)
tabla_diferencia

,equipo,goles_a_favor,goles_en_contra,diferencia_goles
0,Brazil,246,110,136
1,Germany,243,135,108
2,France,149,87,62
3,Argentina,160,102,58
4,Italy,128,77,51
5,Netherlands,107,57,50
6,Spain,116,75,41
7,England,112,71,41
8,Hungary,87,57,30
9,Portugal,69,43,26


## Conclusiones del EDA

- El equipo anfitrión gana significativamente más que el resto de locales.
- Brasil y Alemania lideran históricamente la diferencia de goles en Copas del Mundo.
- Existen goleadas históricas muy por encima del promedio (detectadas vía IQR).
- El dataset procesado quedó guardado en `data/processed/` para su reutilización.